In [1]:
import numpy as np 
import xarray as xr

#my own classes
import sys
sys.path.append("/Users/michelefilippucci/Prog/github/blocktrack/lib")
import blocktoolbox as BT

import time
start_time = time.time()

In [2]:
%cd /Users/michelefilippucci/Data/IFS/z500_daily/
!ls

/Users/michelefilippucci/Data/IFS/z500_daily
ERA5_2.5x2.5_zg500_daily_1940-2024.nc   IFS_z500_hourly_2020-2024.nc
IFS_z500_daily_2020-2024.nc             IFS_z500_northem_daily_2020-2024.nc
IFS_z500_daily_2020-2024_new.nc         IFS_z500_northem_daily_jja_2020-2024.nc
IFS_z500_daily_2020-2024_regrid.nc      try.nc


In [3]:
print("Starting operation")

#'''
data_dir='/Users/michelefilippucci/Data/IFS/z500_daily/'
fn = data_dir + "IFS_z500_northem_daily_2020-2024.nc"
work_dic='/Users/michelefilippucci/Data/IFS/gr_tracking/'
fn_out_unfilt = work_dic + "IFS_northem_2.5x2.5_zg_DAV_tracked_unfilt_daily_2020-2024.nc"
fn_out = work_dic + "IFS_northem_2.5x2.5_zg_DAV_tracked_daily_2020-2024.nc"
fn_dic_unfilt = work_dic + "dictionaries/IFS_northem_2.5x2.5_zg_DAV_tracked_unfilt_daily_2020-2024_DIC.npy"
fn_dic = work_dic + "dictionaries/IFS_northem_2.5x2.5_zg_DAV_tracked_daily_2020-2024_DIC.npy"
'''
data_dir='/Users/michelefilippucci/Data/ERA5/geop/'
fn = data_dir + "ERA5_northem_2.5x2.5_zg500_daily_1960-1963.nc"
work_dic='/Users/michelefilippucci/Data/ERA5/gr_tracking/'
fn_out_unfilt = work_dic + "ERA5_northem_2.5x2.5_zg_DAV_tracked_unfilt_daily_1960-1963.nc"
fn_out = work_dic + "ERA5_northem_2.5x2.5_zg_DAV_tracked_daily_1960-1963.nc"
fn_dic_unfilt = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_DAV_tracked_unfilt_daily_1960-1963_DIC.npy"
fn_dic = work_dic + "dictionaries/ERA5_northem_2.5x2.5_zg_DAV_tracked_daily_1960-1963_DIC.npy"
'''
ds = xr.load_dataset(fn)
print("Data correctly read")
ds = BT.DAV(ds,mer_gradient_filter = True)
print(np.amax(ds['DAV_freq']))
print("DAV function correctly executed")
ds,dic_unfilt = BT.ContourTracking2D(ds,var_name = "DAV")
#saving temporary data
ds.to_netcdf(fn_out_unfilt)
np.save(fn_dic_unfilt,dic_unfilt)


Starting operation
Data correctly read
__Starting a DAV process__
input: zg500 , mer_gradient_filter = True
<xarray.DataArray 'DAV_freq' ()>
array(11.96)
DAV function correctly executed
__Starting a Tracking process__
input: DAV
connected component analysis


100%|██████████████████████████████████████████████████████████████████| 1773/1773 [00:02<00:00, 702.23it/s]


rearranging indexes


100%|█████████████████████████████████████████████████████████████████| 1773/1773 [00:00<00:00, 2130.81it/s]


calculating characteristics
persistence, track, date:


100%|██████████████████████████████████████████████████████████████████| 1773/1773 [00:04<00:00, 357.58it/s]


area, intensity and longitudinal extent


100%|███████████████████████████████████████████████████████████████████| 1773/1773 [00:45<00:00, 38.93it/s]


number of labels: 4387.0


In [4]:
from importlib import reload
reload(BT)
ds,dic_filt = BT.FilterEvents(ds,dic_unfilt,var_name = "DAV",
                              pers_min = 5,min_avg_area = 5e5,max_avg_dist=1000)
print("ContourTracking2D function correclty executed")

__Starting a Filtering process__
input: DAV_tracked
applying filters


100%|████████████████████████████████████████████████████████████████| 4387/4387 [00:00<00:00, 64537.26it/s]


rearranging indexes


100%|████████████████████████████████████████████████████████████████| 1773/1773 [00:00<00:00, 17087.98it/s]

number of labels: 163


ContourTracking2D function correclty executed


In [5]:
#saving data
np.save(fn_dic,dic_filt)
ds.to_netcdf(fn_out)
print("Data created")
  
print("--- %s seconds ---" % (time.time() - start_time))

Data created
--- 58.34283900260925 seconds ---


In [6]:
dic_filt[0]

{}

In [7]:
dic_filt[2]

{'persistence': 5,
 'area': [94297.28773833366,
  1387669.1259313265,
  2142657.286776417,
  2432562.259099355,
  2742735.7950529377],
 'aspect_ratio': [1.6087614290087207,
  27.0312860576479,
  12.62289492716721,
  12.069035210216237,
  8.263236098062077],
 'distance_traveled': [1057.1826998360182,
  540.7094516619748,
  313.3557196449524,
  357.4410479531501],
 'track': ([52.5,
   55.9375,
   58.490566037735846,
   60.03968253968254,
   62.91666666666667],
  [-166.25,
   178.59375,
   170.9433962264151,
   175.55555555555557,
   178.62179487179486]),
 'date': numpy.datetime64('2020-02-15T11:30:00.000000000'),
 'intensity': [368.57560564692085,
  308.683974985488,
  199.03280546331607,
  182.71333335920662,
  127.20479146489043],
 'time': 25}